### Joint P + B analytic covariance validation (cov3)

Compare the analytic (window-based, `compute_covariance_mesh3_spectrum`) joint power spectrum + bispectrum covariance against the mock-based estimate, for LRG 0.4 < z < 0.6. Inputs (produced with `job_scripts/validation_analytic_covariance.py`) are read from `debug_cov3/`.

Notes:
- The first P bin sits at $k = 0$ and the low-$k$ bins are window / integral-constraint dominated: comparisons are restricted to the fit range $0.02 < k < 0.2$.
- Analytic covariances computed before the cov3 PT-term fix (squeezed-trispectrum quadrature guard: relative-azimuth offset + IR-cutoff floor in `jaxpower.cov3`) have a broken BB block (negative diagonals, $\sim 10^9 \times$ off) whenever the binning extends to $k \sim 0$ --- rerun them before trusting the B rows here.

In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

import lsstypes as types

dirname = Path('..') / '..' / 'debug_cov3'
region = 'NGC'

covariance = types.read(dirname / f'covariance_mesh3_spectrum_poles_LRG_z0.4-0.6_{region}_weight-default-FKP.h5')
mocks = types.read(dirname / 'covariance_mocks.h5')
window2 = types.read(dirname / f'window_covariance_mesh2_correlation_LRG_z0.4-0.6_{region}_weight-default-FKP.h5')
window3 = types.read(dirname / f'window_covariance_mesh3_correlation_LRG_z0.4-0.6_{region}_weight-default-FKP.h5')

p_obs, b_obs = list(covariance.observable)
k_p = np.asarray(p_obs.get(p_obs.ells[0]).coords('k'))
k_b = np.asarray(b_obs.get(b_obs.ells[0]).coords('k'))  # (nbins, 2) diagonal pairs
nk, nb = k_p.size, k_b.shape[0]
value, value_mocks = np.asarray(covariance.value()), np.asarray(mocks.value())
print(f'P: {len(p_obs.ells)} ells x {nk} bins | B: {len(b_obs.ells)} ells x {nb} bins | total {value.shape[0]}')

# Flat indices of the fit range, per block
krange = (0.02, 0.2)
index_p = {ell: ill * nk + np.flatnonzero((k_p >= krange[0]) & (k_p <= krange[1])) for ill, ell in enumerate(p_obs.ells)}
offset = len(p_obs.ells) * nk
index_b = {ell: offset + ill * nb + np.flatnonzero((k_b[:, 0] >= krange[0]) & (k_b[:, 0] <= krange[1])) for ill, ell in enumerate(b_obs.ells)}
index_all = np.concatenate(list(index_p.values()) + list(index_b.values()))

#### Covariance windows

Two-anchor $Q_{W,\ell}(s)$ multipoles and the diagonal of the three-anchor $Q_W(s_1, s_2)$; sanity: smooth, decaying, no tail blow-up (they are zero-filled outside the measured separation range).

In [ ]:
fig, lax = plt.subplots(1, 2, figsize=(10., 3.5))
for label, leaf in window2.items(level=None):
    sizes = (len(label['fields1']), len(label['fields2']))
    s = leaf.coords('s')
    lax[0].plot(s, np.abs(np.asarray(leaf.value())), label=f"$\\ell = {label['ells']}$, groups {sizes}")
lax[0].set_xscale('log'); lax[0].set_yscale('log')
lax[0].set_xlabel(r'$s$ [$\mathrm{Mpc}/h$]'); lax[0].set_ylabel(r'$|Q_{W,\ell}(s)|$'); lax[0].legend(fontsize='x-small', ncols=2)
for label, leaf in window3.items(level=None):
    s = leaf.coords('s1')
    lax[1].plot(s, np.abs(np.diag(np.asarray(leaf.value()))), label=f"$\\ell = {label['ells']}$")
lax[1].set_xscale('log'); lax[1].set_yscale('log')
lax[1].set_xlabel(r'$s$ [$\mathrm{Mpc}/h$]'); lax[1].set_ylabel(r'$|Q_W(s, s)|$'); lax[1].legend()
plt.show()

#### Diagonal: power spectrum block

In [ ]:
fig, lax = plt.subplots(2, len(p_obs.ells), sharex=True, figsize=(11., 5.), height_ratios=(3, 1))
for ill, ell in enumerate(p_obs.ells):
    idx = index_p[ell]
    k = k_p[idx - ill * nk]
    std_a, std_m = np.sqrt(np.diag(value)[idx]), np.sqrt(np.diag(value_mocks)[idx])
    lax[0, ill].plot(k, k**2 * std_a, label='analytic')
    lax[0, ill].plot(k, k**2 * std_m, label='mocks')
    lax[0, ill].set_title(rf'$\ell = {ell}$')
    lax[0, ill].set_ylabel(rf'$k^2 \sigma[P_{ell}]$')
    lax[1, ill].plot(k, std_a / std_m, color='k')
    lax[1, ill].axhline(1., color='k', linestyle=':')
    lax[1, ill].set_ylim(0.8, 1.2)
    lax[1, ill].set_ylabel('ratio'); lax[1, ill].set_xlabel(r'$k$ [$h/\mathrm{Mpc}$]')
lax[0, 0].legend()
plt.tight_layout(); plt.show()

#### Diagonal: bispectrum block

In [ ]:
fig, lax = plt.subplots(2, len(b_obs.ells), sharex=True, figsize=(9., 5.), height_ratios=(3, 1))
for ill, ell in enumerate(b_obs.ells):
    idx = index_b[ell]
    k = k_b[idx - offset - ill * nb, 0]
    diag_a, diag_m = np.diag(value)[idx], np.diag(value_mocks)[idx]
    neg = diag_a < 0
    if neg.any(): print(f'ell={ell}: {neg.sum()} negative analytic diagonal entries (broken PT term, see header)')
    lax[0, ill].plot(k, k**4 * np.sqrt(np.abs(diag_a)), label='analytic (|.|)' if neg.any() else 'analytic')
    lax[0, ill].plot(k, k**4 * np.sqrt(diag_m), label='mocks')
    lax[0, ill].set_yscale('log')
    lax[0, ill].set_title(rf'$\ell = {ell}$')
    lax[0, ill].set_ylabel(rf'$k^4 \sigma[B_{{{ell[0]}{ell[1]}{ell[2]}}}]$')
    lax[1, ill].plot(k, np.sqrt(np.abs(diag_a) / diag_m), color='k')
    lax[1, ill].axhline(1., color='k', linestyle=':')
    lax[1, ill].set_ylabel('ratio'); lax[1, ill].set_xlabel(r'$k$ [$h/\mathrm{Mpc}$]')
lax[0, 0].legend()
plt.tight_layout(); plt.show()

#### Correlation matrices (fit range, P + B stacked)

In [ ]:
def corrcoef(cov):
    d = np.sqrt(np.abs(np.diag(cov)))
    return cov / np.outer(d, d)

sub_a = value[np.ix_(index_all, index_all)]
sub_m = value_mocks[np.ix_(index_all, index_all)]
fig, lax = plt.subplots(1, 2, figsize=(11., 5.))
for ax, sub, title in zip(lax, [sub_a, sub_m], ['analytic', 'mocks']):
    im = ax.pcolormesh(corrcoef(sub), vmin=-0.3, vmax=0.3, cmap='RdBu_r')
    nsplit = np.cumsum([len(index_p[ell]) for ell in p_obs.ells] + [len(index_b[ell]) for ell in b_obs.ells])[:-1]
    for split in nsplit:
        ax.axhline(split, color='k', lw=0.5); ax.axvline(split, color='k', lw=0.5)
    ax.set_title(title); ax.set_aspect('equal')
fig.colorbar(im, ax=lax, shrink=0.8)
plt.show()

#### Quantitative comparison

$R_\mathrm{inv}$ and $\chi^2$-scatter metrics (eqs. 2.17--2.18 of arXiv:2404.03007) of the analytic covariance against the mock one, on the fit range.

In [ ]:
from scipy import linalg

def cov_metrics(cov_ref, cov_test):
    inv_ref = np.linalg.inv(cov_ref)
    prod = inv_ref @ cov_test
    ndata = len(cov_ref)
    rinv = np.sqrt(np.sum((prod - np.eye(ndata))**2) / ndata)
    chi2_frac = np.trace(prod) / ndata
    return rinv, chi2_frac

# Thin the P bins so the mock covariance inverse stays well-conditioned
# (ndata must be well below the number of mocks)
index_p_thin = {ell: idx[::5] for ell, idx in index_p.items()}
for label, idx in [('P only', np.concatenate(list(index_p_thin.values()))),
                   ('B only', np.concatenate(list(index_b.values()))),
                   ('P + B', np.concatenate(list(index_p_thin.values()) + list(index_b.values())))]:
    sub_a, sub_m = value[np.ix_(idx, idx)], value_mocks[np.ix_(idx, idx)]
    try:
        rinv, chi2_frac = cov_metrics(sub_m, sub_a)
        print(f'{label:6s}: ndata = {len(idx):4d} | R_inv = {rinv:.3f} | tr(C_mock^-1 C_ana)/n = {chi2_frac:.3f}')
    except np.linalg.LinAlgError as exc:
        print(f'{label:6s}: {exc}')